# Exploratory Data Analysis on the Marathos Dataset

## Loading the dataset

In [0]:
# Use the raw dataset inside the volume
VOLUME_PATH = "/Volumes/marathos_catalog/default/raw/"

# Use spark and sql
spark.sql(f"LIST '{VOLUME_PATH}'")

In [0]:
# See data path
DATA_PATH = "/Volumes/marathos_catalog/default/raw/data/"

display(spark.sql(f"LIST '{DATA_PATH}'"))

In [0]:
# Show what is inside the data folder. Using this method for better debugging later.
FILE_PATH = "/Volumes/marathos_catalog/default/raw/data/TWO_CENTURIES_OF_UM_RACES.csv"

# read csv
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

display(df)

In [0]:
# Check the schema that Spark inferred from the CSV file | column name + data type

df.printSchema()

## Number of rows and columns

In [0]:
row_count = df.count()
column_count = len(df.columns)

print(f"Number of rows: {row_count}")
print(f"Number of columnss: {column_count}")


## Name of columns and number of distinct values

In [0]:
from pyspark.sql.functions import countDistinct

distinct_counts = []

for column in df.columns:
    distinct_count = df.agg(countDistinct(column).alias("distinct_values")).collect()[0]["distinct_values"]
    distinct_counts.append((column, distinct_count))

distinct_counts_df = spark.createDataFrame(
    distinct_counts,
    ["column_name", "distinct_values"]
)

display(distinct_counts_df)

## Descriptive Summary of numerical fields

In [0]:
display(df.describe())

In [0]:
# Create numerical columns for analysis

from pyspark.sql.functions import (
    col,
    regexp_extract,
    regexp_replace,
    split,
    when
)

eda_df = (
    df
    # Extract numeric value from Event distance/length, for example:
    # "50km" -> 50
    # "100mi" -> 100
    # "6h" -> 6
    .withColumn(
        "event_distance_length_value",
        regexp_extract(col("Event distance/length"), r"([0-9]+(?:\.[0-9]+)?)", 1).cast("double")
    )

    # Extract unit from Event distance/length, for example:
    # "50km" -> km
    # "100mi" -> mi
    # "6h" -> h
    .withColumn(
        "event_distance_length_unit",
        regexp_extract(col("Event distance/length"), r"([a-zA-Z]+)", 1)
    )

    # Convert Athlete average speed to numeric
    # Uses try_cast to handle invalid values gracefully (returns null instead of error)
    .withColumn(
        "athlete_average_speed_numeric",
        regexp_replace(col("Athlete average speed"), ",", ".").cast("double")
    )

    # Create athlete age from event year and birth year
    .withColumn(
        "athlete_age",
        col("Year of event") - col("Athlete year of birth")
    )
)

In [0]:

# BASIC EDA: Select meaningful numerical columns

numerical_columns = [
    "Year of event",
    "event_distance_length_value",
    "Event number of finishers",
    "Athlete year of birth",
    "athlete_age",
    "athlete_average_speed_numeric",
    "Athlete ID"
]

display(eda_df.select([try_cast(col, 'double') for col in numerical_columns]).describe())